In [33]:
import json
import requests
import pickle

# Access data and API key
with open('test.pkl', 'rb') as file:
    data = pickle.load(file)

with open("key.json", "r") as file:
    config = json.load(file)

API_KEY = config.get("google_api_key")

import requests

import requests

def get_closest_streetview_and_check_if_outdoor(target_lat, target_lon, api_key):
    try:
        # Street View Metadata API URL
        metadata_url = f"https://maps.googleapis.com/maps/api/streetview/metadata?location={target_lat},{target_lon}&key={api_key}"
        
        # Reverse Geocoding API URL
        geocode_url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={target_lat},{target_lon}&key={api_key}"

        # Request Street View metadata
        metadata_response = requests.get(metadata_url, timeout=10)
        metadata_response.raise_for_status()  # Raise exception for non-200 responses

        metadata = metadata_response.json()
        if metadata.get("status") != "OK":
            return {"status": "error", "message": f"Street View API error: {metadata.get('status')}"}

        # Extract pano ID and coordinates
        pano_id = metadata.get("pano_id")
        closest_location = metadata.get("location", {})

        # Request reverse geocoding to get location description
        geocode_response = requests.get(geocode_url, timeout=10)
        geocode_response.raise_for_status()

        geocode_data = geocode_response.json()
        if geocode_data.get("status") != "OK":
            return {"status": "error", "message": f"Geocode API error: {geocode_data.get('status')}"}

        # Extract address and location types
        address = geocode_data["results"][0].get("formatted_address", "No address available")
        location_types = geocode_data["results"][0].get("types", [])

        # Check for indoor types
        likely_indoor = any(
            indoor_type in location_types for indoor_type in [
                "establishment", "shopping_mall", "store", "gym", "museum", "hospital",
                "library", "school", "university", "lodging", "restaurant", "bar",
                "night_club", "place_of_worship", "spa", "bakery", "casino", "bank",
                "pharmacy", "dentist", "doctor", "travel_agency", "car_dealer", 
                "home_goods_store", "movie_theater", "bowling_alley", "amusement_park", 
                "conference_center", "train_station", "airport", "stadium", "zoo"
            ]
        )

        # Return all results
        return {
            "status": "success",
            "closest_coordinates": closest_location,
            "pano_id": pano_id,
            "description": address,
            "likely_indoor": likely_indoor
        }

    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"Network error: {str(e)}"}
    except KeyError as e:
        return {"status": "error", "message": f"Missing data error: {str(e)}"}
    except Exception as e:
        return {"status": "error", "message": f"Unexpected error: {str(e)}"}


def get_image(latitude, longitude, size, heading, pitch, fov, index):
    # Build the URL
    url = f"https://maps.googleapis.com/maps/api/streetview?size={size}&location={latitude},{longitude}&heading={heading}&pitch={pitch}&fov={fov}&key={API_KEY}"

    # Send the request to Google
    response = requests.get(url)

    image_path = rf"C:\Users\rkhaz\Documents\Drive\try\{index}.jpg"

    # Check the response
    if response.status_code == 200:
        # Save the image
        with open(image_path, "wb") as file:
            file.write(response.content)
        print(f"Street View image saved as {index}.jpg'")

        return image_path

    else:
        print(f"Error: {response.status_code}, {response.text}")

In [34]:
for index, row in data[:100].iterrows():

    size = "640x640"
    pitch = 0  # Camera tilt (0 = straight ahead)
    fov = 90   # Field of view
    heading = 0

    lat = row['lat_lng'][0]
    lng = row['lat_lng'][1]

    result = get_closest_streetview_and_check_if_outdoor(lat, lng, API_KEY)

    if result["status"] == "success":

        if result['likely_indoor'] == True:
            print(result["description"], result['likely_indoor'])

            get_image(lat, lng, size, heading, pitch, fov, index)





Hazelaarweg 39, 7556 DN Hengelo, Netherlands True
Street View image saved as 1248954.jpg'
Haagdoorn 4, 6226 WN Maastricht, Netherlands True
Street View image saved as 1248996.jpg'
Kerkpad 7, 5085 EK Esbeek, Netherlands True
Street View image saved as 1249015.jpg'
